# DSCI 552 - Homework 3
### Xinyi Wu

## 1. Time Series Classification Part 1: Feature Creation/Extraction

### 1(a) Download and Load the AReM Dataset

In [38]:
import os
import zipfile
import pandas as pd
import numpy as np

zip_path = "AReM.zip"
extract_path = "AReM"

# Unzip the dataset if it has not already been extracted
if not os.path.exists(extract_path):
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extract_path)

data_path = os.path.join("AReM", "AReM")

print("Data path:", data_path)
print("Activities:", os.listdir(data_path))

Data path: AReM/AReM
Activities: ['walking', 'bending2', 'sitting', 'sensorsPlacement.pdf', 'standing', 'cycling', 'bending1', 'bendingType.pdf', 'lying', '.DS_Store']


In [39]:
data_path = os.path.join("AReM", "AReM")

activities = [
    "bending1",
    "bending2",
    "cycling",
    "lying",
    "sitting",
    "standing",
    "walking"
]

total = 0

for activity in activities:
    folder = os.path.join(data_path, activity)

    csv_files = [
        f for f in os.listdir(folder)
        if f.endswith(".csv")
    ]

    print(f"{activity}: {len(csv_files)} datasets")
    total += len(csv_files)

print("\nTotal:", total)

bending1: 7 datasets
bending2: 6 datasets
cycling: 15 datasets
lying: 15 datasets
sitting: 15 datasets
standing: 15 datasets
walking: 15 datasets

Total: 88


### 1(b) Train/Test Split

In [40]:
train_files = []
test_files = []

for activity in activities:
    folder = os.path.join(data_path, activity)

    csv_files = sorted(
        [f for f in os.listdir(folder) if f.endswith(".csv")],
        key=lambda x: int(x.replace("dataset", "").replace(".csv", ""))
    )

    for file in csv_files:
        dataset_num = int(file.replace("dataset", "").replace(".csv", ""))

        if activity in ["bending1", "bending2"]:
            is_test = dataset_num in [1, 2]
        else:
            is_test = dataset_num in [1, 2, 3]

        record = (activity, file)

        if is_test:
            test_files.append(record)
        else:
            train_files.append(record)

print("Training instances:", len(train_files))
print("Test instances:", len(test_files))
print("Total:", len(train_files) + len(test_files))

print("\nTest set:")
for item in test_files:
    print(item)

Training instances: 69
Test instances: 19
Total: 88

Test set:
('bending1', 'dataset1.csv')
('bending1', 'dataset2.csv')
('bending2', 'dataset1.csv')
('bending2', 'dataset2.csv')
('cycling', 'dataset1.csv')
('cycling', 'dataset2.csv')
('cycling', 'dataset3.csv')
('lying', 'dataset1.csv')
('lying', 'dataset2.csv')
('lying', 'dataset3.csv')
('sitting', 'dataset1.csv')
('sitting', 'dataset2.csv')
('sitting', 'dataset3.csv')
('standing', 'dataset1.csv')
('standing', 'dataset2.csv')
('standing', 'dataset3.csv')
('walking', 'dataset1.csv')
('walking', 'dataset2.csv')
('walking', 'dataset3.csv')


### 1(c)(i) Time-Domain Features

Common time-domain features used in time series classification include:

- Minimum
- Maximum
- Mean
- Median
- Standard deviation
- Variance
- First quartile
- Third quartile
- Interquartile range
- Range
- Root mean square (RMS)
- Skewness
- Kurtosis
- Zero-crossing rate

In [41]:
import pandas as pd

sample_path = os.path.join(data_path, "bending1", "dataset1.csv")

with open(sample_path, "r") as f:
    for i in range(10):
        print(repr(f.readline()))

'# Task: bending1\n'
'# Frequency (Hz): 20\n'
'# Clock (millisecond): 250\n'
'# Duration (seconds): 120\n'
'# Columns: time,avg_rss12,var_rss12,avg_rss13,var_rss13,avg_rss23,var_rss23\n'
'0,39.25,0.43,22.75,0.43,33.75,1.30\n'
'250,39.25,0.43,23.00,0.00,33.00,0.00\n'
'500,39.25,0.43,23.25,0.43,33.00,0.00\n'
'750,39.50,0.50,23.00,0.71,33.00,0.00\n'
'1000,39.50,0.50,24.00,0.00,33.00,0.00\n'


In [42]:
columns = [
    "time",
    "avg_rss12",
    "var_rss12",
    "avg_rss13",
    "var_rss13",
    "avg_rss23",
    "var_rss23"
]

df = pd.read_csv(
    sample_path,
    skiprows=5,
    header=None,
    names=columns
)

print("Shape:", df.shape)
print()
print(df.head())

Shape: (480, 7)

   time  avg_rss12  var_rss12  avg_rss13  var_rss13  avg_rss23  var_rss23
0     0      39.25       0.43      22.75       0.43      33.75        1.3
1   250      39.25       0.43      23.00       0.00      33.00        0.0
2   500      39.25       0.43      23.25       0.43      33.00        0.0
3   750      39.50       0.50      23.00       0.71      33.00        0.0
4  1000      39.50       0.50      24.00       0.00      33.00        0.0


In [43]:
import numpy as np

time_series_columns = [
    "avg_rss12",
    "var_rss12",
    "avg_rss13",
    "var_rss13",
    "avg_rss23",
    "var_rss23"
]

def extract_features(df):
    features = {}

    for i, col in enumerate(time_series_columns, start=1):
        series = pd.to_numeric(df[col], errors="coerce").dropna()

        features[f"min{i}"] = series.min()
        features[f"max{i}"] = series.max()
        features[f"mean{i}"] = series.mean()
        features[f"median{i}"] = series.median()
        features[f"std{i}"] = series.std()
        features[f"1st_quart{i}"] = series.quantile(0.25)
        features[f"3rd_quart{i}"] = series.quantile(0.75)

    return features

In [44]:
import io

def read_arem_file(file_path):
    # Read raw text
    with open(file_path, "r") as f:
        lines = f.readlines()

    # Remove trailing commas from data lines if present
    cleaned_lines = []
    for line in lines:
        line = line.rstrip("\n")

        if line.endswith(","):
            line = line[:-1]

        cleaned_lines.append(line)

    cleaned_text = "\n".join(cleaned_lines)

    df = pd.read_csv(
        io.StringIO(cleaned_text),
        skiprows=5,
        header=None,
        names=columns
    )

    return df

### 1(c)(ii) Extract Time-Domain Features

In [45]:
all_features = []

for activity in activities:
    folder = os.path.join(data_path, activity)

    csv_files = sorted(
        [f for f in os.listdir(folder) if f.endswith(".csv")],
        key=lambda x: int(x.replace("dataset", "").replace(".csv", ""))
    )

    for file in csv_files:
        file_path = os.path.join(folder, file)

        # Read and clean the dataset
        temp_df = read_arem_file(file_path)

        # Extract 42 time-domain features
        features = extract_features(temp_df)

        # Keep activity and dataset names
        features["activity"] = activity
        features["dataset"] = file

        all_features.append(features)

feature_df = pd.DataFrame(all_features)

print("Number of instances:", feature_df.shape[0])
print("Number of extracted features:", feature_df.shape[1] - 2)
print("Total columns including activity and dataset:", feature_df.shape[1])

feature_df.head()

Number of instances: 88
Number of extracted features: 42
Total columns including activity and dataset: 44


,min1,max1,mean1,median1,std1,1st_quart1,3rd_quart1,min2,max2,mean2,...,3rd_quart5,min6,max6,mean6,median6,std6,1st_quart6,3rd_quart6,activity,dataset
0,37.25,45.00,40.624792,40.50,1.476967,39.25,42.00,0.0,1.30,0.358604,...,36.00,0.0,1.92,0.570583,0.43,0.582915,0.0,1.30,bending1,dataset1.csv
1,38.00,45.67,42.812812,42.50,1.435550,42.00,43.67,0.0,1.22,0.372437,...,34.50,0.0,3.11,0.571083,0.43,0.601010,0.0,1.30,bending1,dataset2.csv
2,35.00,47.40,43.954500,44.33,1.558835,43.00,45.00,0.0,1.70,0.426250,...,36.50,0.0,1.79,0.493292,0.43,0.513506,0.0,0.94,bending1,dataset3.csv
3,33.00,47.75,42.179812,43.50,3.670666,39.15,45.00,0.0,3.00,0.696042,...,36.33,0.0,2.18,0.613521,0.50,0.524317,0.0,1.00,bending1,dataset4.csv
4,33.00,45.75,41.678063,41.75,2.243490,41.33,42.75,0.0,2.83,0.535979,...,31.25,0.0,1.79,0.383292,0.43,0.389164,0.0,0.50,bending1,dataset5.csv


### 1(c)(iii) Standard Deviations and 90% Bootstrap Confidence Intervals

In [46]:
feature_columns = [
    col for col in feature_df.columns
    if col not in ["activity", "dataset"]
]

feature_std = feature_df[feature_columns].std()

print("Number of features:", len(feature_std))
print()
print(feature_std)

Number of features: 42

min1          9.624011
max1          4.207745
mean1         5.276431
median1       5.386624
std1          1.771282
1st_quart1    6.127846
3rd_quart1    5.031028
min2          0.000000
max2          5.059656
mean2         1.577908
median2       1.413545
std2          0.885875
1st_quart2    0.948434
3rd_quart2    2.131337
min3          2.954516
max3          4.819848
mean3         3.976754
median3       4.009687
std3          0.952011
1st_quart3    4.184148
3rd_quart3    4.153451
min4          0.000000
max4          2.181809
mean4         1.168268
median4       1.148591
std4          0.458612
1st_quart4    0.844919
3rd_quart4    1.555235
min5          6.121205
max5          5.773355
mean5         5.704025
median5       5.844368
std5          1.012358
1st_quart5    6.122142
3rd_quart5    5.563553
min6          0.046101
max6          2.533515
mean6         1.157222
median6       1.089858
std6          0.516968
1st_quart6    0.761542
3rd_quart6    1.526871
dtype: flo

In [47]:
np.random.seed(42)

bootstrap_results = []

for feature in feature_columns:
    values = feature_df[feature].dropna().values

    bootstrap_stds = []

    for _ in range(1000):
        sample = np.random.choice(
            values,
            size=len(values),
            replace=True
        )

        bootstrap_stds.append(
            np.std(sample, ddof=1)
        )

    lower = np.percentile(bootstrap_stds, 5)
    upper = np.percentile(bootstrap_stds, 95)

    bootstrap_results.append({
        "Feature": feature,
        "Standard Deviation": np.std(values, ddof=1),
        "90% CI Lower": lower,
        "90% CI Upper": upper
    })

bootstrap_df = pd.DataFrame(bootstrap_results)

bootstrap_df

,Feature,Standard Deviation,90% CI Lower,90% CI Upper
0,min1,9.624011,8.321447,10.822204
1,max1,4.207745,3.054311,5.118384
2,mean1,5.276431,4.600504,5.861422
3,median1,5.386624,4.709493,5.982450
4,std1,1.771282,1.579549,1.944076
5,1st_quart1,6.127846,5.520521,6.626887
6,3rd_quart1,5.031028,4.198566,5.760116
7,min2,0.000000,0.000000,0.000000
8,max2,5.059656,4.615051,5.409296
9,mean2,1.577908,1.418010,1.711959


### 1(c)(iv) Selection of Important Time-Domain Features

The three most important time-domain features I selected are **minimum, mean, and maximum**.

The mean represents the overall level of a time series, while the minimum and maximum capture its lower and upper extremes. Together, these features provide a simple representation of the central tendency and range of the sensor measurements.

## 2. ISLR 3.7.4

### (a)
The cubic regression will have training RSS that is less than or equal to the training RSS of the linear regression. The cubic model contains the linear model as a special case when the coefficients of the squared and cubic terms are zero. Therefore, adding these extra predictors cannot increase the training RSS, even though the true relationship is linear.

### (b)
The linear regression is expected to have a lower test RSS. Since the true relationship is linear, the additional squared and cubic terms do not represent the true relationship and may cause the cubic model to overfit the training data. Therefore, the simpler linear model is expected to generalize better to test data.

### (c)
The cubic regression will have training RSS that is less than or equal to the training RSS of the linear regression. Since the cubic model is more flexible and contains the linear model as a special case, it can fit the training observations at least as well as the linear model.

### (d)
There is not enough information to determine which model will have lower test RSS. The true relationship is nonlinear, but we do not know its form or how far it is from linear. If the true relationship is well approximated by a cubic function, the cubic model may have lower test RSS. However, if the nonlinearity is small or has a different form, the linear model may perform better because the cubic model can overfit.